# APC Feature Store & Training Datasets
*Co-authored with CoCo*

Refactors the ad-hoc CSV/INSERT ML pipeline (`app/ml_insights.py`) into a proper Snowflake Feature Store with:
- **Entities**: MATERIAL, COST_COMPONENT_PERIOD
- **Feature Views**: Product risk features, anomaly features, forecast features
- **Training Datasets**: Point-in-time correct datasets for model retraining

In [ ]:
# Session setup
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql_simplifier_enabled = True
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
# Create the Feature Store schema
session.sql("CREATE SCHEMA IF NOT EXISTS APC_DEPLOY_DB.ML_FEATURE_STORE").collect()
print("Schema APC_DEPLOY_DB.ML_FEATURE_STORE ready")

In [ ]:
# Initialize the Feature Store
from snowflake.ml.feature_store import FeatureStore, CreationMode, Entity, FeatureView

fs = FeatureStore(
    session=session,
    database="APC_DEPLOY_DB",
    name="ML_FEATURE_STORE",
    default_warehouse="APC_DEPLOY_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Feature Store initialized")

In [ ]:
# ── Entity: MATERIAL ──
# The primary entity for product-level risk scores and forecasts
material_entity = Entity(
    name="MATERIAL",
    join_keys=["MATERIAL_NUMBER"],
    desc="SAP material (product) — join key for risk scores, cost variance, and profitability features"
)
fs.register_entity(material_entity)
print("Registered entity: MATERIAL")

In [ ]:
# ── Entity: COST_COMPONENT_PERIOD ──
# Composite entity for anomaly detection at the component x period grain
cost_component_period_entity = Entity(
    name="COST_COMPONENT_PERIOD",
    join_keys=["COST_COMPONENT", "FISCAL_YEAR", "PERIOD"],
    desc="Cost component in a fiscal period — join key for anomaly detection features"
)
fs.register_entity(cost_component_period_entity)
print("Registered entity: COST_COMPONENT_PERIOD")

# Verify
fs.list_entities().show()

In [ ]:
# ── Feature View 1: PRODUCT_RISK_FV ──
# Per-material risk features derived from FY2026 variance trajectory
# Replaces the PRODUCT_RISK_SCORES table from ml_insights.py

risk_features_df = session.sql("""
    WITH period_data AS (
        SELECT
            MATERIAL_NUMBER,
            MAX(MATERIAL_DESCRIPTION) AS MATERIAL_DESCRIPTION,
            MAX(CASE WHEN PERIOD = '001' THEN COST_VARIANCE_PCT END) AS VAR_P001,
            MAX(CASE WHEN PERIOD = '003' THEN COST_VARIANCE_PCT END) AS VAR_P003,
            MAX(CASE WHEN PERIOD = '006' THEN COST_VARIANCE_PCT END) AS VAR_P006,
            REGR_SLOPE(COST_VARIANCE_PCT, CAST(PERIOD AS INT)) AS VARIANCE_SLOPE,
            COUNT(*) AS PERIODS_OBSERVED,
            AVG(COST_VARIANCE_PCT) AS AVG_VARIANCE,
            STDDEV(COST_VARIANCE_PCT) AS VARIANCE_VOLATILITY,
            MAX(ABS(COST_VARIANCE_PCT)) AS MAX_ABS_VARIANCE
        FROM APC_DEPLOY_DB.ANALYTICS.PRODUCT_COST_SUMMARY
        WHERE FISCAL_YEAR = 2026 AND MATERIAL_TYPE = 'FERT'
        GROUP BY MATERIAL_NUMBER
    )
    SELECT
        MATERIAL_NUMBER,
        MATERIAL_DESCRIPTION,
        ROUND(VAR_P001, 4) AS VAR_P001,
        ROUND(VAR_P003, 4) AS VAR_P003,
        ROUND(VAR_P006, 4) AS VAR_P006,
        ROUND(VARIANCE_SLOPE, 4) AS VARIANCE_SLOPE,
        PERIODS_OBSERVED,
        ROUND(AVG_VARIANCE, 4) AS AVG_VARIANCE,
        ROUND(VARIANCE_VOLATILITY, 4) AS VARIANCE_VOLATILITY,
        ROUND(MAX_ABS_VARIANCE, 4) AS MAX_ABS_VARIANCE,
        ROUND(COALESCE(VAR_P006, VAR_P003, VAR_P001, 0) + COALESCE(VARIANCE_SLOPE, 0), 4) AS ESTIMATED_NEXT_PERIOD,
        CASE
            WHEN COALESCE(VAR_P006, 0) > 8 OR (COALESCE(VAR_P006, 0) + COALESCE(VARIANCE_SLOPE, 0)) > 10 THEN 'HIGH'
            WHEN COALESCE(VAR_P006, 0) > 5 OR (COALESCE(VAR_P006, 0) + COALESCE(VARIANCE_SLOPE, 0)) > 7 THEN 'MEDIUM'
            ELSE 'LOW'
        END AS RISK_LEVEL
    FROM period_data
""")

risk_fv = FeatureView(
    name="PRODUCT_RISK_FV",
    entities=[material_entity],
    feature_df=risk_features_df,
    refresh_freq="1 day",
    desc="Product risk features: variance trajectory, slope, volatility, and risk classification"
)

registered_risk_fv = fs.register_feature_view(feature_view=risk_fv, version="V1", block=True)
print("Registered: PRODUCT_RISK_FV V1")

In [ ]:
# ── Feature View 2: ANOMALY_DETECTION_FV ──
# Per-component-period anomaly features (replaces COST_ANOMALIES logic)

anomaly_features_df = session.sql("""
    SELECT
        COST_COMPONENT,
        FISCAL_YEAR,
        PERIOD,
        ROUND(AVG(COMPONENT_VARIANCE_PCT), 4) AS MEAN_VARIANCE,
        ROUND(STDDEV(COMPONENT_VARIANCE_PCT), 4) AS STD_VARIANCE,
        ROUND(MAX(ABS(COMPONENT_VARIANCE_PCT)), 4) AS MAX_ABS_VARIANCE,
        ROUND(MEDIAN(COMPONENT_VARIANCE_PCT), 4) AS MEDIAN_VARIANCE,
        COUNT(*) AS RECORD_COUNT,
        ROUND(MAX(COMPONENT_VARIANCE_PCT) - MIN(COMPONENT_VARIANCE_PCT), 4) AS VARIANCE_RANGE,
        ROUND(AVG(ACTUAL_COST), 2) AS AVG_ACTUAL_COST,
        ROUND(AVG(STANDARD_COST), 2) AS AVG_STANDARD_COST
    FROM APC_DEPLOY_DB.ANALYTICS.COST_COMPONENT_DETAIL
    GROUP BY COST_COMPONENT, FISCAL_YEAR, PERIOD
""")

anomaly_fv = FeatureView(
    name="ANOMALY_DETECTION_FV",
    entities=[cost_component_period_entity],
    feature_df=anomaly_features_df,
    refresh_freq="1 day",
    desc="Anomaly detection features: statistical aggregates per cost component per period"
)

registered_anomaly_fv = fs.register_feature_view(feature_view=anomaly_fv, version="V1", block=True)
print("Registered: ANOMALY_DETECTION_FV V1")

In [ ]:
# ── Feature View 3: VARIANCE_FORECAST_FV ──
# Per-material time-series features for forecasting (with timestamp for PIT joins)

forecast_features_df = session.sql("""
    SELECT
        MATERIAL_NUMBER,
        DATE_FROM_PARTS(FISCAL_YEAR, CAST(PERIOD AS INT), 1) AS TS,
        ROUND(AVG(COST_VARIANCE_PCT), 4) AS VARIANCE_PCT,
        ROUND(AVG(ACTUAL_COST_PER_UNIT), 4) AS AVG_ACTUAL_COST,
        ROUND(AVG(STANDARD_COST_PER_UNIT), 4) AS AVG_STANDARD_COST,
        COUNT(DISTINCT PLANT_CODE) AS PLANT_COUNT,
        ROUND(STDDEV(COST_VARIANCE_PCT), 4) AS CROSS_PLANT_VOLATILITY
    FROM APC_DEPLOY_DB.ANALYTICS.PRODUCT_COST_SUMMARY
    WHERE MATERIAL_TYPE = 'FERT'
    GROUP BY MATERIAL_NUMBER, DATE_FROM_PARTS(FISCAL_YEAR, CAST(PERIOD AS INT), 1)
""")

forecast_fv = FeatureView(
    name="VARIANCE_FORECAST_FV",
    entities=[material_entity],
    feature_df=forecast_features_df,
    timestamp_col="TS",
    refresh_freq="1 day",
    desc="Time-series features for variance forecasting: per-material monthly cost metrics"
)

registered_forecast_fv = fs.register_feature_view(feature_view=forecast_fv, version="V1", block=True)
print("Registered: VARIANCE_FORECAST_FV V1")

In [ ]:
# ── Verify all registered feature views ──
print("\n=== Feature Store Contents ===")
print("\nEntities:")
fs.list_entities().show()
print("\nFeature Views:")
fs.list_feature_views().select("NAME", "VERSION", "DESC", "SCHEDULING_STATE").show()

## Training Datasets

Generate versioned, immutable training datasets from the feature views. These provide:
- Point-in-time correct joins (no data leakage)
- Reproducibility (same version = same data)
- Lineage tracking (which features trained which model)

In [ ]:
# ── Training Dataset: Anomaly Detection ──
# Spine: all component x period combinations to classify

anomaly_spine_df = session.sql("""
    SELECT DISTINCT COST_COMPONENT, FISCAL_YEAR, PERIOD
    FROM APC_DEPLOY_DB.ANALYTICS.COST_COMPONENT_DETAIL
""")

anomaly_training_dataset = fs.generate_dataset(
    name="ANOMALY_TRAINING_DS",
    version="V1",
    spine_df=anomaly_spine_df,
    features=[registered_anomaly_fv],
    desc="Training dataset for Isolation Forest anomaly detection on cost component aggregates",
)

df = anomaly_training_dataset.read.to_pandas()
print(f"Anomaly training dataset shape: {df.shape}")
df.head()

In [ ]:
# ── Training Dataset: Risk Scoring ──
# Spine: all finished products in FY2026

risk_spine_df = session.sql("""
    SELECT DISTINCT MATERIAL_NUMBER
    FROM APC_DEPLOY_DB.ANALYTICS.PRODUCT_COST_SUMMARY
    WHERE FISCAL_YEAR = 2026 AND MATERIAL_TYPE = 'FERT'
""")

risk_training_dataset = fs.generate_dataset(
    name="RISK_SCORING_DS",
    version="V1",
    spine_df=risk_spine_df,
    features=[registered_risk_fv],
    desc="Training dataset for product risk classification (HIGH/MEDIUM/LOW)",
)

df = risk_training_dataset.read.to_pandas()
print(f"Risk training dataset shape: {df.shape}")
df.head()

In [ ]:
# ── Training Dataset: Forecasting (point-in-time correct) ──
# Spine: each material at each month

forecast_spine_df = session.sql("""
    SELECT DISTINCT
        MATERIAL_NUMBER,
        DATE_FROM_PARTS(FISCAL_YEAR, CAST(PERIOD AS INT), 1) AS TS
    FROM APC_DEPLOY_DB.ANALYTICS.PRODUCT_COST_SUMMARY
    WHERE MATERIAL_TYPE = 'FERT'
    ORDER BY MATERIAL_NUMBER, TS
""")

forecast_training_dataset = fs.generate_dataset(
    name="FORECAST_TRAINING_DS",
    version="V1",
    spine_df=forecast_spine_df,
    features=[registered_forecast_fv],
    spine_timestamp_col="TS",
    desc="Point-in-time correct training dataset for variance forecasting",
)

df = forecast_training_dataset.read.to_pandas()
print(f"Forecast training dataset shape: {df.shape}")
df.head()

In [ ]:
# ── Summary ──
print("="*70)
print("Feature Store setup complete!")
print("="*70)
print(f"\nDatabase: APC_DEPLOY_DB")
print(f"Schema:   ML_FEATURE_STORE")
print(f"\nEntities:")
print(f"  - MATERIAL (join_key: MATERIAL_NUMBER)")
print(f"  - COST_COMPONENT_PERIOD (join_keys: COST_COMPONENT, FISCAL_YEAR, PERIOD)")
print(f"\nFeature Views (auto-refreshing Dynamic Tables):")
print(f"  - PRODUCT_RISK_FV V1: variance slope, volatility, risk level")
print(f"  - ANOMALY_DETECTION_FV V1: statistical aggregates for Isolation Forest")
print(f"  - VARIANCE_FORECAST_FV V1: time-series features with PIT support")
print(f"\nTraining Datasets (immutable, versioned):")
print(f"  - ANOMALY_TRAINING_DS V1: for anomaly detection model")
print(f"  - RISK_SCORING_DS V1: for risk classification model")
print(f"  - FORECAST_TRAINING_DS V1: for time-series forecasting (PIT correct)")
print(f"\nUsage:")
print(f"  df = fs.generate_dataset(...).read.to_pandas()  # get training data")
print(f"  fs.get_feature_view('PRODUCT_RISK_FV', 'V1')    # retrieve a view")
print(f"  Feature views auto-refresh daily via Dynamic Tables")

## Model Training & Registration

Train three models from the Feature Store datasets and register them in the Snowflake Model Registry:
1. **Isolation Forest** — anomaly detection on cost component aggregates
2. **Random Forest Classifier** — product risk scoring (HIGH/MEDIUM/LOW)
3. **Holt-Winters** — time-series variance forecasting per material

In [ ]:
# ── Model 1: Isolation Forest for Anomaly Detection ──
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Load training data from Feature Store dataset
anomaly_df = anomaly_training_dataset.read.to_pandas()

# Feature columns (exclude entity keys)
feature_cols = ['MEAN_VARIANCE', 'STD_VARIANCE', 'MAX_ABS_VARIANCE',
                'MEDIAN_VARIANCE', 'RECORD_COUNT', 'VARIANCE_RANGE',
                'AVG_ACTUAL_COST', 'AVG_STANDARD_COST']

X_anomaly = anomaly_df[feature_cols].fillna(0)

# Scale features
scaler_anomaly = StandardScaler()
X_scaled = scaler_anomaly.fit_transform(X_anomaly)

# Train Isolation Forest
iforest = IsolationForest(
    n_estimators=100,
    contamination=0.1,  # expect ~10% anomalies
    random_state=42
)
iforest.fit(X_scaled)

# Score the training data
anomaly_df['ANOMALY_SCORE'] = iforest.decision_function(X_scaled)
anomaly_df['IS_ANOMALY'] = (iforest.predict(X_scaled) == -1).astype(int)

print(f"Isolation Forest trained on {len(X_anomaly)} rows")
print(f"Anomalies detected: {anomaly_df['IS_ANOMALY'].sum()} ({anomaly_df['IS_ANOMALY'].mean()*100:.1f}%)")
anomaly_df[anomaly_df['IS_ANOMALY'] == 1][['COST_COMPONENT', 'FISCAL_YEAR', 'PERIOD', 'ANOMALY_SCORE']].head()

In [ ]:
# ── Model 2: Random Forest Classifier for Risk Scoring ──
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Load training data from Feature Store dataset
risk_df = risk_training_dataset.read.to_pandas()

# Features and target
risk_feature_cols = ['VAR_P001', 'VAR_P003', 'VAR_P006', 'VARIANCE_SLOPE',
                     'AVG_VARIANCE', 'VARIANCE_VOLATILITY', 'MAX_ABS_VARIANCE',
                     'ESTIMATED_NEXT_PERIOD']

X_risk = risk_df[risk_feature_cols].fillna(0)
y_risk = risk_df['RISK_LEVEL']

# Train Random Forest classifier
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    class_weight='balanced'
)
rf_classifier.fit(X_risk, y_risk)

# Cross-validation score
cv_scores = cross_val_score(rf_classifier, X_risk, y_risk, cv=min(3, len(X_risk)), scoring='accuracy')

print(f"Random Forest trained on {len(X_risk)} products")
print(f"Cross-val accuracy: {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})")
print(f"Feature importances:")
for feat, imp in sorted(zip(risk_feature_cols, rf_classifier.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")

In [ ]:
# ── Model 3: Holt-Winters for Variance Forecasting ──
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# Load time-series training data from Feature Store
forecast_df = forecast_training_dataset.read.to_pandas()
forecast_df['TS'] = pd.to_datetime(forecast_df['TS'])

# Train one model per material (Many-Model pattern)
forecast_models = {}
materials = forecast_df['MATERIAL_NUMBER'].unique()

for material in materials:
    mat_df = forecast_df[forecast_df['MATERIAL_NUMBER'] == material].sort_values('TS')
    ts = mat_df.set_index('TS')['VARIANCE_PCT'].asfreq('MS')
    ts = ts.fillna(method='ffill').fillna(0)

    if len(ts) >= 6:
        try:
            model = ExponentialSmoothing(
                ts, trend='add', seasonal=None,
                initialization_method='estimated'
            ).fit(optimized=True)
            forecast_models[material] = model
        except:
            pass

print(f"Holt-Winters models trained for {len(forecast_models)}/{len(materials)} materials")

# Example forecast for first material
first_material = list(forecast_models.keys())[0]
fcast = forecast_models[first_material].forecast(3)
print(f"\n{first_material} — 3-month forecast:")
for date, val in fcast.items():
    print(f"  {date.strftime('%Y-%m')}: {val:.2f}%")

In [ ]:
# ── Register all models in the Snowflake Model Registry ──
import os
os.environ['USE_STREAMLIT_WIDGETS'] = '0'

# Patch: streamlit in container runtime lacks .runtime attribute
import streamlit as st
if not hasattr(st, 'runtime'):
    class _FakeRuntime:
        @staticmethod
        def exists():
            return False
    st.runtime = _FakeRuntime()

from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="APC_DEPLOY_DB", schema_name="ML_FEATURE_STORE")

# Register Isolation Forest
iforest_mv = reg.log_model(
    model=iforest,
    model_name="APC_ANOMALY_DETECTOR",
    version_name="V1",
    conda_dependencies=["scikit-learn"],
    sample_input_data=session.create_dataframe(pd.DataFrame(X_scaled[:5], columns=feature_cols)),
    comment="Isolation Forest for cost component anomaly detection. Trained on ANOMALY_TRAINING_DS V1."
)
print(f"Registered: APC_ANOMALY_DETECTOR V1")

# Register Random Forest Classifier
rf_mv = reg.log_model(
    model=rf_classifier,
    model_name="APC_RISK_CLASSIFIER",
    version_name="V1",
    conda_dependencies=["scikit-learn"],
    sample_input_data=session.create_dataframe(X_risk.head(5)),
    comment="Random Forest risk classifier (HIGH/MEDIUM/LOW). Trained on RISK_SCORING_DS V1."
)
print(f"Registered: APC_RISK_CLASSIFIER V1")

print("\n=== Model Registry ===")
reg.show_models().select("NAME", "COMMENT").show()

In [ ]:
# ── Register Holt-Winters forecast models (as a custom model) ──
from snowflake.ml.model import custom_model
import pickle

class VarianceForecastModel(custom_model.CustomModel):
    """Multi-material Holt-Winters forecast model."""

    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        with open(context.path("models.pkl"), "rb") as f:
            self.models = pickle.load(f)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in input_df.iterrows():
            material = row['MATERIAL_NUMBER']
            horizon = int(row.get('HORIZON', 3))
            if material in self.models:
                fcast = self.models[material].forecast(horizon)
                results.append({
                    'MATERIAL_NUMBER': material,
                    'FORECAST_VALUES': fcast.tolist(),
                    'FORECAST_DATES': [d.strftime('%Y-%m') for d in fcast.index]
                })
            else:
                results.append({
                    'MATERIAL_NUMBER': material,
                    'FORECAST_VALUES': [],
                    'FORECAST_DATES': []
                })
        return pd.DataFrame(results)

# Save models to temp file
import tempfile, os
tmp_dir = tempfile.mkdtemp()
models_path = os.path.join(tmp_dir, 'models.pkl')
with open(models_path, 'wb') as f:
    pickle.dump(forecast_models, f)

# Register custom model
forecast_mv = reg.log_model(
    model=VarianceForecastModel(
        context=custom_model.ModelContext(
            models={"models.pkl": models_path}
        )
    ),
    model_name="APC_VARIANCE_FORECASTER",
    version_name="V1",
    conda_dependencies=["pandas", "statsmodels"],
    sample_input_data=session.create_dataframe(
        pd.DataFrame({'MATERIAL_NUMBER': [materials[0]], 'HORIZON': [3]})
    ),
    comment="Multi-material Holt-Winters variance forecaster. Trained on FORECAST_TRAINING_DS V1."
)
print(f"Registered: APC_VARIANCE_FORECASTER V1")

In [ ]:
# ── Final Summary ──
print("="*70)
print("Model Training & Registration Complete!")
print("="*70)
print(f"\nModel Registry: APC_DEPLOY_DB.ML_FEATURE_STORE")
print(f"\nRegistered Models:")
print(f"  1. APC_ANOMALY_DETECTOR V1")
print(f"     - Algorithm: Isolation Forest (contamination=0.1)")
print(f"     - Training data: ANOMALY_TRAINING_DS V1 ({len(X_anomaly)} rows)")
print(f"     - Anomalies found: {anomaly_df['IS_ANOMALY'].sum()}")
print(f"")
print(f"  2. APC_RISK_CLASSIFIER V1")
print(f"     - Algorithm: Random Forest (n_estimators=100, max_depth=5)")
print(f"     - Training data: RISK_SCORING_DS V1 ({len(X_risk)} products)")
print(f"     - CV accuracy: {cv_scores.mean():.2%}")
print(f"")
print(f"  3. APC_VARIANCE_FORECASTER V1")
print(f"     - Algorithm: Holt-Winters Exponential Smoothing (per material)")
print(f"     - Training data: FORECAST_TRAINING_DS V1 ({len(forecast_df)} rows)")
print(f"     - Materials modelled: {len(forecast_models)}")
print(f"")
print(f"Usage (batch inference):")
print(f"  mv = reg.get_model('APC_RISK_CLASSIFIER').version('V1')")
print(f"  predictions = mv.run(input_df, function_name='predict')")
print(f"")
print(f"Lineage: Training datasets -> Models (tracked automatically)")

## Model Training & Registration

Train models on the Feature Store datasets and register them in the Snowflake Model Registry.

In [ ]:
# ── Train Isolation Forest for Anomaly Detection ──
import pandas as pd
from sklearn.ensemble import IsolationForest
from snowflake.ml.registry import Registry

# Load training data from Feature Store dataset
anomaly_df = fs.read_feature_view(registered_anomaly_fv).to_pandas()
print(f"Anomaly training data: {anomaly_df.shape}")
anomaly_df.head()

In [ ]:
# Train Isolation Forest
feature_cols = ['MEAN_VARIANCE', 'STD_VARIANCE', 'MAX_ABS_VARIANCE', 
                'MEDIAN_VARIANCE', 'RECORD_COUNT', 'VARIANCE_RANGE',
                'AVG_ACTUAL_COST', 'AVG_STANDARD_COST']

X_anomaly = anomaly_df[feature_cols].fillna(0)

iso_forest = IsolationForest(
    contamination=0.1,  # expect ~10% anomalies
    random_state=42,
    n_estimators=100
)
iso_forest.fit(X_anomaly)

# Predict: -1 = anomaly, 1 = normal
anomaly_df['IS_ANOMALY'] = iso_forest.predict(X_anomaly) == -1
anomaly_df['ANOMALY_SCORE'] = iso_forest.decision_function(X_anomaly)

print(f"Anomalies detected: {anomaly_df['IS_ANOMALY'].sum()} / {len(anomaly_df)}")
anomaly_df[anomaly_df['IS_ANOMALY']].head()

In [ ]:
# Register Isolation Forest in Model Registry
reg = Registry(session=session, database_name="APC_DEPLOY_DB", schema_name="ML_FEATURE_STORE")

mv_anomaly = reg.log_model(
    model=iso_forest,
    model_name="ANOMALY_DETECTION_MODEL",
    version_name="V1",
    sample_input_data=anomaly_df[feature_cols].head(5),
    comment="Isolation Forest trained on Feature Store ANOMALY_DETECTION_FV for cost component anomaly detection",
)
print(f"Registered: ANOMALY_DETECTION_MODEL V1")
print(f"Functions: {mv_anomaly.show_functions()}")

In [ ]:
# ── Train Risk Classifier ──
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# Load risk training data
risk_df = fs.read_feature_view(registered_risk_fv).to_pandas()
print(f"Risk training data: {risk_df.shape}")
risk_df[['MATERIAL_NUMBER', 'VARIANCE_SLOPE', 'AVG_VARIANCE', 'RISK_LEVEL']]

In [ ]:
# Train Gradient Boosting Classifier for risk level prediction
risk_feature_cols = ['VAR_P001', 'VAR_P003', 'VAR_P006', 'VARIANCE_SLOPE',
                     'AVG_VARIANCE', 'VARIANCE_VOLATILITY', 'MAX_ABS_VARIANCE',
                     'ESTIMATED_NEXT_PERIOD']

X_risk = risk_df[risk_feature_cols].fillna(0)
y_risk = risk_df['RISK_LEVEL']

le = LabelEncoder()
y_encoded = le.fit_transform(y_risk)

risk_clf = GradientBoostingClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=42
)
risk_clf.fit(X_risk, y_encoded)

# Feature importance
importances = pd.DataFrame({
    'feature': risk_feature_cols,
    'importance': risk_clf.feature_importances_
}).sort_values('importance', ascending=False)
print("Feature Importance:")
print(importances.to_string(index=False))
print(f"\nTraining accuracy: {risk_clf.score(X_risk, y_encoded):.2%}")

In [ ]:
# Register Risk Classifier in Model Registry
mv_risk = reg.log_model(
    model=risk_clf,
    model_name="RISK_CLASSIFICATION_MODEL",
    version_name="V1",
    sample_input_data=risk_df[risk_feature_cols].head(5),
    comment="Gradient Boosting classifier for product risk level (HIGH/MEDIUM/LOW) trained on Feature Store PRODUCT_RISK_FV",
)
print(f"Registered: RISK_CLASSIFICATION_MODEL V1")
print(f"Functions: {mv_risk.show_functions()}")

In [ ]:
# ── Verify registered models ──
models = reg.show_models()
print("\n=== Model Registry Contents ===")
for _, m in models.iterrows():
    print(f"  {m['name']}")

print("\n=== Anomaly Model - Test Prediction ===")
test_pred = mv_anomaly.run(anomaly_df[feature_cols].head(3), function_name="predict")
print(test_pred)

print("\n=== Risk Model - Test Prediction ===")
test_risk = mv_risk.run(risk_df[risk_feature_cols].head(3), function_name="predict")
print(test_risk)

## Forecast Models — Cost Variance & Sales Revenue

Train XGBoost time-series regressors on the Feature Store datasets and register them in the Model Registry. These replace the built-in `SNOWFLAKE.ML.FORECAST` with versioned, traceable models.

In [ ]:
# Re-establish session and registry (in case kernel restarted)
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.registry import Registry
import pandas as pd
import numpy as np

session = get_active_session()
fs = FeatureStore(session=session, database="APC_DEPLOY_DB", name="ML_FEATURE_STORE",
                  default_warehouse="APC_DEPLOY_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
reg = Registry(session=session, database_name="APC_DEPLOY_DB", schema_name="ML_FEATURE_STORE")
print("Session, Feature Store, and Registry ready")

In [ ]:
# ── Cost Variance Forecast Model ──
# Load time-series features from the Feature Store
variance_df = session.sql("""
    SELECT MATERIAL_NUMBER, TS, VARIANCE_PCT, AVG_ACTUAL_COST, AVG_STANDARD_COST,
           PLANT_COUNT, CROSS_PLANT_VOLATILITY
    FROM APC_DEPLOY_DB.ML_FEATURE_STORE."VARIANCE_FORECAST_FV$V1"
    ORDER BY MATERIAL_NUMBER, TS
""").to_pandas()

# Engineer lag features for time-series prediction
def add_lag_features(df, target_col, group_col, lags=[1, 2, 3]):
    df = df.sort_values([group_col, 'TS']).copy()
    for lag in lags:
        df[f'{target_col}_LAG{lag}'] = df.groupby(group_col)[target_col].shift(lag)
    df[f'{target_col}_ROLLING3'] = df.groupby(group_col)[target_col].transform(
        lambda x: x.rolling(3, min_periods=1).mean())
    return df

variance_df = add_lag_features(variance_df, 'VARIANCE_PCT', 'MATERIAL_NUMBER')
variance_df['MONTH'] = pd.to_datetime(variance_df['TS']).dt.month
variance_df = variance_df.dropna(subset=['VARIANCE_PCT_LAG1'])

print(f"Cost variance training data: {variance_df.shape}")
variance_df.head()

In [ ]:
# Train XGBoost regressor for cost variance forecasting
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

feature_cols_var = ['VARIANCE_PCT_LAG1', 'VARIANCE_PCT_LAG2', 'VARIANCE_PCT_LAG3',
                    'VARIANCE_PCT_ROLLING3', 'AVG_ACTUAL_COST', 'AVG_STANDARD_COST',
                    'PLANT_COUNT', 'CROSS_PLANT_VOLATILITY', 'MONTH']

X_var = variance_df[feature_cols_var]
y_var = variance_df['VARIANCE_PCT']

# Train/test split: last 2 months as test
split_date = variance_df['TS'].max() - pd.DateOffset(months=2)
train_mask = pd.to_datetime(variance_df['TS']) <= split_date

X_train, X_test = X_var[train_mask], X_var[~train_mask]
y_train, y_test = y_var[train_mask], y_var[~train_mask]

variance_model = XGBRegressor(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=42, objective='reg:squarederror'
)
variance_model.fit(X_train, y_train)

y_pred = variance_model.predict(X_test)
print(f"Cost Variance Forecast Model:")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred):.4f}")
print(f"  Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# Register Cost Variance Forecast model
mv_variance_forecast = reg.log_model(
    model=variance_model,
    model_name="COST_VARIANCE_FORECAST_MODEL",
    version_name="V1",
    sample_input_data=variance_df[feature_cols_var].head(5),
    comment="XGBoost regressor for per-material cost variance forecasting. Trained on VARIANCE_FORECAST_FV with lag features.",
)
print("Registered: COST_VARIANCE_FORECAST_MODEL V1")

In [ ]:
# ── Sales Revenue Forecast Model ──
# Load sales billing data and build time-series features
sales_df = session.sql("""
    SELECT MATNR AS MATERIAL_NUMBER,
           DATE_FROM_PARTS(GJAHR, CAST(POPER AS INT), 1) AS TS,
           SUM(NETWR) AS REVENUE,
           SUM(FKMNG) AS VOLUME,
           AVG(NETPR) AS AVG_PRICE
    FROM APC_DEPLOY_DB.SAP_BDC.SD_BILLING
    GROUP BY MATNR, DATE_FROM_PARTS(GJAHR, CAST(POPER AS INT), 1)
    ORDER BY MATNR, TS
""").to_pandas()

# Engineer lag features for revenue
sales_df = add_lag_features(sales_df, 'REVENUE', 'MATERIAL_NUMBER')
sales_df['MONTH'] = pd.to_datetime(sales_df['TS']).dt.month
sales_df['VOLUME_LAG1'] = sales_df.groupby('MATERIAL_NUMBER')['VOLUME'].shift(1)
sales_df['AVG_PRICE_LAG1'] = sales_df.groupby('MATERIAL_NUMBER')['AVG_PRICE'].shift(1)
sales_df = sales_df.dropna(subset=['REVENUE_LAG1'])

print(f"Sales revenue training data: {sales_df.shape}")
sales_df.head()

In [ ]:
# Train XGBoost regressor for sales revenue forecasting
feature_cols_sales = ['REVENUE_LAG1', 'REVENUE_LAG2', 'REVENUE_LAG3',
                      'REVENUE_ROLLING3', 'VOLUME', 'AVG_PRICE',
                      'VOLUME_LAG1', 'AVG_PRICE_LAG1', 'MONTH']

X_sales = sales_df[feature_cols_sales]
y_sales = sales_df['REVENUE']

# Train/test split
split_date_sales = sales_df['TS'].max() - pd.DateOffset(months=1)
train_mask_sales = pd.to_datetime(sales_df['TS']) <= split_date_sales

X_train_s, X_test_s = X_sales[train_mask_sales], X_sales[~train_mask_sales]
y_train_s, y_test_s = y_sales[train_mask_sales], y_sales[~train_mask_sales]

sales_model = XGBRegressor(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=42, objective='reg:squarederror'
)
sales_model.fit(X_train_s, y_train_s)

y_pred_s = sales_model.predict(X_test_s)
print(f"Sales Revenue Forecast Model:")
print(f"  MAE:  ${mean_absolute_error(y_test_s, y_pred_s):,.2f}")
print(f"  R²:   {r2_score(y_test_s, y_pred_s):.4f}")
print(f"  Train: {len(X_train_s)}, Test: {len(X_test_s)}")

In [ ]:
# Register Sales Revenue Forecast model
# Fill nulls with 0 for the sample (lag3 is null for materials with < 4 periods)
sample_data = sales_df[feature_cols_sales].fillna(0).head(5)
print(f"Sample data for signature: {sample_data.shape}")

mv_sales_forecast = reg.log_model(
    model=sales_model,
    model_name="SALES_REVENUE_FORECAST_MODEL",
    version_name="V1",
    sample_input_data=sample_data,
    comment="XGBoost regressor for per-material sales revenue forecasting. Trained on SD_BILLING with lag features.",
)
print("Registered: SALES_REVENUE_FORECAST_MODEL V1")

In [ ]:
# ── Final verification ──
print("\n" + "="*70)
print("ALL REGISTERED MODELS")
print("="*70)
models = reg.show_models()
for _, m in models.iterrows():
    print(f"  {m['name']:40s} (default: {m['default_version_name']})")
print(f"\nTotal: {len(models)} models in APC_DEPLOY_DB.ML_FEATURE_STORE")